# LUAD training

This public notebook was migrated from the audited read-only research source. Configure paths in `config.yaml` before execution. Time is metadata and is never a model input.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'LUAD'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import re
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_2d as s1
from stvirtual.models import stage2_2d_lineage as s2


In [ ]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS)
diff_path = str(DIFF_MAP)


In [ ]:
route_ids=['AAH', 'LUAD']
fracs = ['AAH_to_LUAD']
seg_key = "AAH_to_LUAD" 
steps = 100


In [ ]:
adata = sc.read_h5ad(SCANVI_ADATA)
adata


In [ ]:
adata.obs['status'].unique()


In [ ]:
model = None


In [ ]:
importlib.reload(s1)

res1 = s1.train_model_multislice(
    model=model,                  
    adata_all=adata,
    slice_key="status",
    route_ids=route_ids,
    save_root=str(STAGE1_CHECKPOINT_ROOT),
    x_key="cx_aligned",
    y_key="cy_aligned",
    latent_key="X_scanVI",    
    steps=steps,  
    guide_eps=0.01,
    uot_eps=0.05, uot_tau=0.1, uot_lam_x=1.0, uot_lam_f=0.001,
    guide_topk=32, guide_temp=0.5, guide_schedule="linear",
    terrain_gap=False,
    cell_type_key='_clone_s',
    lam_context=0.0,
    lam_residual=1.0,
    lam_vsmooth=0.1,  
    lam_uot=1.0,
    lib_layer='counts',              
    latent_dim=10,             
    epochs=100,
    device="cuda:0",
)


In [ ]:
saved = {'AAH_to_LUAD': str(CHECKPOINT_ROOT / 'stage1_res' / 'rollout_stage1_AAH_to_LUAD.npz')}


## Save Stage-1 traces and generate 2D boundaries

In [ ]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import rollout_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="status", ann_key="_clone_s",
    save_prefix="rollout_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_key in fracs:
    source_name = stage_key.split("_to_", 1)[0]
    match = re.search(r"\d+", source_name)
    source_number = int(match.group(0)) if match else 10**9
    trace = rollout_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + source_number + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


In [ ]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=SCANVI_ADATA,
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / "AAH_to_LUAD" / "checkpoints" / "best.pt"),
    device=device,
    layer_col="_clone_s",
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            diff_csv=diff_path,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key="X_scanVI",
            z_csv_offset=0,
            layer_col="_clone_s",  
        )
    )

outs = s2.run_multi_stages(
    ctx=ctx,
    sample_key = 'status',
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=10, LR=1e-4),
    rl_xy=2.0,
    rl_z=0.01,
)

for o in outs:
    print(o["best_ckpt_path"])   


In [ ]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


In [ ]:
importlib.reload(s2)
rollouts = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    rollouts[seg_key] = s2.rollout_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key],sample_key='status',
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'rollout' / seg_key, output_prefix=seg_key,  
    )
    print(seg_key, "Tp1=", len(rollouts[seg_key]["coords"]))


In [ ]:
r0 = rollouts['AAH_to_LUAD']
print(r0.keys())


In [ ]:
import numpy as np
import pandas as pd

fi = 1           # 

layers = np.asarray(r0["layers"][fi]).astype(int)
is_diff = np.asarray(r0["is_diff"][fi]).astype(bool)
diff_tgt = np.asarray(r0["diff_tgt_layer"][fi]).astype(int)
commit = np.asarray(r0["commit_step"][fi]).astype(int)

#  diff ：is_diff=True, tgt>=0,  commit
m = is_diff & (diff_tgt >= 0) & (commit < 0)

df = pd.DataFrame({
    "src_layer_internal": layers[m],
    "tgt_layer_internal": diff_tgt[m],
})

print("n_active_diff =", len(df))
print(df.value_counts().sort_index())


In [ ]:
ro = rollouts[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")


In [ ]:
ro.keys()


## Persist lineage metadata in each H5AD frame

Write UID-based identity and lineage fields into each saved frame. AnnData row names are set to UID so cross-frame tracking remains stable after cell birth and death.


In [ ]:
LINEAGE_OBS_FIELDS = (
    "uid",
    "parent_uid",
    "diff_alpha",
    "src_layer_id",
    "src_layer",
    "tgt_layer_id",
    "tgt_layer",
)

def persist_lineage_rollout_obs(rollout_result, celltype_names):
    output_paths = rollout_result.get("output_paths")
    required = ("uid", "parent_uid", "layers", "is_diff", "diff_alpha", "diff_tgt_layer", "commit_step")
    missing = [field for field in required if not isinstance(rollout_result.get(field), list)]
    if missing:
        raise KeyError(f"rollout_result is missing lineage fields: {missing}")
    if not isinstance(output_paths, list):
        raise KeyError("rollout_result has no output_paths; pass output_dir to rollout_policy_one_stage")
    frame_count = len(rollout_result["coords"])
    if len(output_paths) != frame_count:
        raise ValueError("output_paths and rollout frames have different lengths")

    names = [str(name) for name in celltype_names]
    source_by_uid = {}
    target_by_uid = {}
    for frame_index in range(frame_count):
        uids = np.asarray(rollout_result["uid"][frame_index], dtype=np.int64)
        current_layers = np.asarray(rollout_result["layers"][frame_index], dtype=np.int64)
        target_layers = np.asarray(rollout_result["diff_tgt_layer"][frame_index], dtype=np.int64)
        for uid, current_layer, target_layer in zip(uids, current_layers, target_layers):
            source_by_uid.setdefault(int(uid), int(current_layer))
            if int(target_layer) >= 0:
                target_by_uid[int(uid)] = int(target_layer)

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        uids = np.asarray(rollout_result["uid"][frame_index], dtype=np.int64)
        parent_uids = np.asarray(rollout_result["parent_uid"][frame_index], dtype=np.int64)
        diff_alpha = np.asarray(rollout_result["diff_alpha"][frame_index], dtype=np.float32)
        is_diff = np.asarray(rollout_result["is_diff"][frame_index], dtype=bool)
        commit_step = np.asarray(rollout_result["commit_step"][frame_index], dtype=np.int32)
        has_lineage = is_diff | (commit_step >= 0)

        src_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        tgt_ids = np.full(frame_adata.n_obs, -1, dtype=np.int64)
        for index, uid in enumerate(uids):
            if has_lineage[index]:
                src_ids[index] = source_by_uid[int(uid)]
                tgt_ids[index] = target_by_uid.get(int(uid), -1)

        src_names = np.array([names[index] if 0 <= index < len(names) else "" for index in src_ids], dtype=str)
        tgt_names = np.array([names[index] if 0 <= index < len(names) else "" for index in tgt_ids], dtype=str)
        frame_adata.obs["uid"] = uids
        frame_adata.obs_names = uids.astype(str)
        frame_adata.obs["parent_uid"] = parent_uids
        frame_adata.obs["diff_alpha"] = diff_alpha
        frame_adata.obs["src_layer_id"] = src_ids
        frame_adata.obs["src_layer"] = src_names
        frame_adata.obs["tgt_layer_id"] = tgt_ids
        frame_adata.obs["tgt_layer"] = tgt_names
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": frame_count, "obs_fields": list(LINEAGE_OBS_FIELDS)}

rollout_obs_reports = {
    route: persist_lineage_rollout_obs(route_rollout, ctx.layers_list)
    for route, route_rollout in rollouts.items()
}
rollout_obs_reports
